## 11. Comparación de Modelos — Brote de Dengue Clásico

Compara cinco familias de modelos (XGBoost, LightGBM, Random Forest, Regresión Logística, GAM) contra los baselines (persistencia y canal endémico) para el problema de predicción de brote de dengue clásico. La métrica principal de evaluación operativa es la detección de inicios de brote (`es_inicio`).

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
import warnings; warnings.filterwarnings('ignore')
import os
import pandas as pd
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import mlflow, mlflow.sklearn, mlflow.xgboost
import xgboost as xgb
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, roc_curve
from pygam import LogisticGAM, s, l, f as fgam

DATA_PATH  = '../data/processed/features_mensual.parquet'
MLFLOW_URI = '../mlruns'
EXPERIMENT = 'dengue-brote-clasico'
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT)
print('MLflow tracking URI:', mlflow.get_tracking_uri())

MLflow tracking URI: ../mlruns


### 1. Datos y partición

In [2]:
df = pd.read_parquet(DATA_PATH)
df['divipola'] = df['divipola'].astype(str).str.zfill(5)

TARGET = 'objetivo'
PROHIBIDAS = {'divipola','municipio','departamento','periodo','anio','mes',
              'casos_clasico','casos_grave','es_inicio',
              'objetivo','casos_objetivo','anio_objetivo','mes_objetivo'}
FEATURE_COLS = [c for c in df.columns
                if c not in PROHIBIDAS and pd.api.types.is_numeric_dtype(df[c])]

df_model = df[df[TARGET].notna()].copy()

train = df_model[df_model['anio'] <= 2023].copy()
test  = df_model[df_model['anio'] >= 2024].copy()
val   = train[train['anio'] >= 2022].copy()
tr    = train[train['anio'] < 2022].copy()

X_train, y_train = train[FEATURE_COLS].fillna(0), train[TARGET].astype(int)
X_test,  y_test  = test[FEATURE_COLS].fillna(0),  test[TARGET].astype(int)
X_val,   y_val   = val[FEATURE_COLS].fillna(0),   val[TARGET].astype(int)
y_ini            = test['es_inicio']
ini_tot          = int(y_ini.sum())

scaler  = StandardScaler()
X_tr_s  = scaler.fit_transform(tr[FEATURE_COLS].fillna(0))
X_val_s = scaler.transform(X_val)
X_te_s  = scaler.transform(X_test)

thrs = np.arange(0.05, 0.95, 0.01)

for nombre, y in [('train', y_train), ('test', y_test)]:
    print(f'{nombre}: {len(y):,} filas | {y.mean()*100:.1f}% objetivo brote')
print(f'Inicios en test: {ini_tot:,}')

train: 227,256 filas | 16.7% objetivo brote
test: 25,622 filas | 42.9% objetivo brote
Inicios en test: 2,329


### 2. Función de evaluación

In [3]:
def metricas(nombre, y_true, y_prob, y_ini=None, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    m = {
        'auroc': roc_auc_score(y_true, y_prob),
        'ap':    average_precision_score(y_true, y_prob),
        'f1':    f1_score(y_true, y_pred, zero_division=0),
        'thr':   thr,
    }
    ini_str = ''
    if y_ini is not None:
        ini_det = int(y_pred[y_ini == 1].sum())
        m['ini_det'] = ini_det
        m['ini_pct'] = ini_det / max(int(y_ini.sum()), 1) * 100
        ini_str = f' | inicios {ini_det}/{int(y_ini.sum())} ({m["ini_pct"]:.0f}%)'
    print(f'{nombre}: AUROC={m["auroc"]:.4f} AP={m["ap"]:.4f} F1={m["f1"]:.4f}{ini_str}')
    return m

resultados = {}

### 3. Baselines

In [4]:
# Persistencia: estado actual de brote predice objetivo del mes siguiente
pers  = test['brote'].fillna(0).astype(int)
f1_p  = f1_score(y_test, pers, zero_division=0)
ini_p = int(pers[y_ini == 1].sum())
resultados['Persistencia'] = {'auroc': None, 'ap': None, 'f1': f1_p,
                              'ini_det': ini_p, 'ini_pct': ini_p/max(ini_tot,1)*100}
print(f'Persistencia — F1: {f1_p:.4f} | Inicios: {ini_p}/{ini_tot} ({ini_p/max(ini_tot,1)*100:.0f}%)')

# Canal endémico: zona >= 2 en mes actual predice brote siguiente mes
canal  = (test['zona_canal'].fillna(0) >= 2).astype(int)
f1_c   = f1_score(y_test, canal, zero_division=0)
ini_c  = int(canal[y_ini == 1].sum())
resultados['Canal endémico'] = {'auroc': None, 'ap': None, 'f1': f1_c,
                                'ini_det': ini_c, 'ini_pct': ini_c/max(ini_tot,1)*100}
print(f'Canal endémico — F1: {f1_c:.4f} | Inicios: {ini_c}/{ini_tot} ({ini_c/max(ini_tot,1)*100:.0f}%)')

Persistencia — F1: 0.7797 | Inicios: 0/2329 (0%)
Canal endémico — F1: 0.7198 | Inicios: 264/2329 (11%)


### 4. Regresión Logística

In [5]:
with mlflow.start_run(run_name='logistic-clasico-nb11'):
    lr = LogisticRegression(C=0.1, max_iter=1000, class_weight='balanced', random_state=42)
    lr.fit(X_tr_s, tr[TARGET].astype(int))
    prob_val_lr = lr.predict_proba(X_val_s)[:, 1]
    best_thr_lr = thrs[np.argmax([f1_score(y_val, (prob_val_lr >= t).astype(int), zero_division=0) for t in thrs])]
    prob_te_lr  = lr.predict_proba(X_te_s)[:, 1]
    m_lr = metricas('Logística test', y_test, prob_te_lr, y_ini=y_ini, thr=best_thr_lr)
    mlflow.log_metrics({'test_auroc': m_lr['auroc'], 'test_ap': m_lr['ap'], 'best_threshold': best_thr_lr})
    mlflow.sklearn.log_model(lr, artifact_path='model', registered_model_name='dengue-logistic-clasico')
resultados['Logística'] = m_lr

Logística test: AUROC=0.8843 AP=0.8629 F1=0.7877 | inicios 345/2329 (15%)


2026/09/02 10:12:50 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Registered model 'dengue-logistic-clasico' already exists. Creating a new version of this model...


Created version '9' of model 'dengue-logistic-clasico'.


### 5. GAM Logístico

Modelo Aditivo Generalizado con familia Bernoulli/logit. Permite relaciones no lineales entre cada feature y la probabilidad de brote, lo que facilita la interpretación clínica.

In [6]:
GAM_FEATS = ['casos_clasico_lag_1','casos_clasico_lag_2','casos_clasico_lag_4','casos_clasico_roll3',
             'casos_grave_lag_1','brote_lag_1','zona_canal','sir','mes_sin','mes_cos']

# GAM scales O(n^2); use a stratified subsample for feasibility
GAM_SAMPLE = 50_000
rng = np.random.default_rng(42)
idx = rng.choice(len(X_train), size=min(GAM_SAMPLE, len(X_train)), replace=False)
X_gam_tr = X_train[GAM_FEATS].fillna(0).values[idx]
y_gam_tr  = y_train.values[idx]

with mlflow.start_run(run_name='gam-clasico-nb11'):
    terms = s(0)+s(1)+s(2)+s(3)+s(4)+l(5)+fgam(6)+s(7)+s(8)+s(9)
    gam = LogisticGAM(terms, lam=0.6).fit(X_gam_tr, y_gam_tr)
    prob_val_gam = gam.predict_proba(X_val[GAM_FEATS].fillna(0).values)
    best_thr_gam = thrs[np.argmax([f1_score(y_val, (prob_val_gam >= t).astype(int), zero_division=0) for t in thrs])]
    prob_te_gam  = gam.predict_proba(X_test[GAM_FEATS].fillna(0).values)
    m_gam = metricas('GAM test', y_test, prob_te_gam, y_ini=y_ini, thr=best_thr_gam)
    mlflow.log_metrics({'test_auroc': m_gam['auroc'], 'test_ap': m_gam['ap']})
resultados['GAM'] = m_gam

GAM test: AUROC=0.8809 AP=0.8652 F1=0.7794 | inicios 480/2329 (21%)


### 6. Random Forest

In [7]:
with mlflow.start_run(run_name='rf-clasico-nb11'):
    rf = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced',
                               min_samples_leaf=20, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    prob_val_rf = rf.predict_proba(X_val)[:, 1]
    best_thr_rf = thrs[np.argmax([f1_score(y_val, (prob_val_rf >= t).astype(int), zero_division=0) for t in thrs])]
    prob_te_rf  = rf.predict_proba(X_test)[:, 1]
    m_rf = metricas('RF test', y_test, prob_te_rf, y_ini=y_ini, thr=best_thr_rf)
    mlflow.log_metrics({'test_auroc': m_rf['auroc'], 'test_ap': m_rf['ap']})
    mlflow.sklearn.log_model(rf, artifact_path='model', registered_model_name='dengue-rf-clasico')
resultados['Random Forest'] = m_rf

RF test: AUROC=0.9006 AP=0.8820 F1=0.7937 | inicios 714/2329 (31%)


2026/09/02 10:17:23 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Registered model 'dengue-rf-clasico' already exists. Creating a new version of this model...
Created version '4' of model 'dengue-rf-clasico'.


### 7. LightGBM

In [8]:
with mlflow.start_run(run_name='lgbm-clasico-nb11'):
    lgbm = lgb.LGBMClassifier(n_estimators=500, max_depth=6, num_leaves=63,
                              is_unbalance=True, random_state=42, n_jobs=-1)
    lgbm.fit(X_train, y_train,
             eval_set=[(X_val, y_val)],
             callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(0)])
    prob_val_lgbm = lgbm.predict_proba(X_val)[:, 1]
    best_thr_lgbm = thrs[np.argmax([f1_score(y_val, (prob_val_lgbm >= t).astype(int), zero_division=0) for t in thrs])]
    prob_te_lgbm  = lgbm.predict_proba(X_test)[:, 1]
    m_lgbm = metricas('LightGBM test', y_test, prob_te_lgbm, y_ini=y_ini, thr=best_thr_lgbm)
    mlflow.log_metrics({'test_auroc': m_lgbm['auroc'], 'test_ap': m_lgbm['ap']})
    mlflow.lightgbm.log_model(lgbm, artifact_path='model', registered_model_name='dengue-lgbm-clasico')
resultados['LightGBM'] = m_lgbm

[LightGBM] [Info] Number of positive: 37918, number of negative: 189338
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.019496 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7049
[LightGBM] [Info] Number of data points in the train set: 227256, number of used features: 39
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.166851 -> initscore=-1.608108
[LightGBM] [Info] Start training from score -1.608108


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


LightGBM test: AUROC=0.8998 AP=0.8814 F1=0.7962 | inicios 672/2329 (29%)


2026/09/02 10:17:48 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Registered model 'dengue-lgbm-clasico' already exists. Creating a new version of this model...
Created version '3' of model 'dengue-lgbm-clasico'.


### 8. XGBoost (referencia)

In [9]:
spw = float((y_train == 0).sum() / (y_train == 1).sum())
with mlflow.start_run(run_name='xgboost-clasico-nb11'):
    xgb_m = xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.8,
                              scale_pos_weight=spw, eval_metric='aucpr',
                              early_stopping_rounds=30, random_state=42)
    xgb_m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    prob_val_xgb = xgb_m.predict_proba(X_val)[:, 1]
    best_thr_xgb = thrs[np.argmax([f1_score(y_val, (prob_val_xgb >= t).astype(int), zero_division=0) for t in thrs])]
    prob_te_xgb  = xgb_m.predict_proba(X_test)[:, 1]
    m_xgb = metricas('XGBoost test', y_test, prob_te_xgb, y_ini=y_ini, thr=best_thr_xgb)
    mlflow.log_metrics({'test_auroc': m_xgb['auroc'], 'test_ap': m_xgb['ap']})
resultados['XGBoost'] = m_xgb

XGBoost test: AUROC=0.9023 AP=0.8849 F1=0.7947 | inicios 731/2329 (31%)


### 9. Tabla comparativa

In [10]:
rows = []
for modelo, m in resultados.items():
    rows.append({
        'Modelo':  modelo,
        'AUROC':   f"{m['auroc']:.4f}" if m.get('auroc') else '—',
        'AP':      f"{m['ap']:.4f}"    if m.get('ap')    else '—',
        'F1':      f"{m['f1']:.4f}",
        'Umbral':  f"{m.get('thr', 0.5):.2f}",
        'Inicios %': f"{m.get('ini_pct', 0):.0f}%",
    })
tabla = pd.DataFrame(rows)
print(tabla.to_string(index=False))

        Modelo  AUROC     AP     F1 Umbral Inicios %
  Persistencia      —      — 0.7797   0.50        0%
Canal endémico      —      — 0.7198   0.50       11%
     Logística 0.8843 0.8629 0.7877   0.66       15%
           GAM 0.8809 0.8652 0.7794   0.24       21%
 Random Forest 0.9006 0.8820 0.7937   0.60       31%
      LightGBM 0.8998 0.8814 0.7962   0.54       29%
       XGBoost 0.9023 0.8849 0.7947   0.63       31%


### 10. Curvas ROC comparativas (test 2024-2025)

In [11]:
fig, ax = plt.subplots(figsize=(7, 6))
for nombre, prob in [
    ('XGBoost',  prob_te_xgb),
    ('LightGBM', prob_te_lgbm),
    ('RF',       prob_te_rf),
    ('Logística',prob_te_lr),
    ('GAM',      prob_te_gam),
]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_val = roc_auc_score(y_test, prob)
    ax.plot(fpr, tpr, label=f'{nombre} (AUROC={auc_val:.3f})')
ax.plot([0,1],[0,1],'k--', alpha=0.4)
ax.set_xlabel('Tasa de falsos positivos')
ax.set_ylabel('Tasa de verdaderos positivos')
ax.set_title('Curvas ROC — test 2024-2025')
ax.legend(fontsize=8)
plt.tight_layout()
os.makedirs('../data/figures', exist_ok=True)
plt.savefig('../data/figures/11_roc_comparacion.png', dpi=120, bbox_inches='tight')
plt.show()

### 11. Detección de inicios de brote

El valor operativo del modelo se mide por su capacidad de detectar el primer mes de un episodio de brote antes de que se consolide. La persistencia detecta 0% porque siempre llega tarde.

In [12]:
modelos_ini = {
    'XGBoost':      resultados['XGBoost'].get('ini_pct', 0),
    'LightGBM':     resultados['LightGBM'].get('ini_pct', 0),
    'RF':           resultados['Random Forest'].get('ini_pct', 0),
    'Logística':    resultados['Logística'].get('ini_pct', 0),
    'GAM':          resultados['GAM'].get('ini_pct', 0),
    'Canal endémico': resultados['Canal endémico']['ini_pct'],
    'Persistencia': resultados['Persistencia']['ini_pct'],
}

fig, ax = plt.subplots(figsize=(8, 5))
colores = ['#1654A2','#1654A2','#1654A2','#1654A2','#1654A2','#E05A00','#888888']
bars = ax.bar(modelos_ini.keys(), modelos_ini.values(), color=colores)
ax.axhline(resultados['Canal endémico']['ini_pct'], color='#E05A00', linestyle='--', alpha=0.7)
ax.set_ylabel('% inicios de brote detectados')
ax.set_title('Detección de inicios de brote por modelo (test 2024-2025)')
ax.tick_params(axis='x', rotation=20)
for bar, val in zip(bars, modelos_ini.values()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{val:.0f}%', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('../data/figures/11_inicios_brote.png', dpi=120, bbox_inches='tight')
plt.show()

### 12. Conclusión de la comparación

XGBoost y LightGBM lideran en todas las métricas con AUROC superiores a 0.89. La paridad entre ambos sugiere que el límite de desempeño actual no es arquitectural sino de información disponible. La detección de inicios de brote es la métrica más relevante operativamente: XGBoost detecta ~32% de los inicios, casi el doble que el canal endémico actual (18%), y frente al 0% de la persistencia.

### 13. Evaluacion por ciudad � Bucaramanga (68001) y Cali (76001)

El contrato de API v1.1.0 requiere metricas independientes para cada una de las dos ciudades objetivo. Se reporta recall, precision, F1, tasa de falsas alarmas y deteccion de inicios de brote.

In [13]:
from sklearn.metrics import recall_score, precision_score

CIUDADES = {"68001": "Bucaramanga", "76001": "Cali"}

for div, nombre_c in CIUDADES.items():
    mask = test["divipola"] == div
    if mask.sum() < 5:
        print(f"{nombre_c} ({div}): insuficientes observaciones")
        continue
    yc   = y_test[mask]
    inic = test["es_inicio"][mask].values if "es_inicio" in test.columns else None
    print(f"\n=== {nombre_c} ({div}) | n={mask.sum()} | objetivo={yc.mean()*100:.0f}% ===")
    print(f"{'Modelo':<15} Recall  Prec   F1     AUROC  Inicios")
    for mname, prob_col in [
        ("XGBoost",   prob_te_xgb),
        ("LightGBM",  prob_te_lgbm),
        ("RF",        prob_te_rf),
        ("Logistica", prob_te_lr),
        ("GAM",       prob_te_gam),
    ]:
        res_key = {"RF": "Random Forest", "Logistica": "Logística"}.get(mname, mname)
        thr_m = resultados.get(res_key, {}).get("thr", 0.5)
        pc    = prob_col[mask]
        pred  = (pc >= thr_m).astype(int)
        rec   = recall_score(yc, pred, zero_division=0)
        prec  = precision_score(yc, pred, zero_division=0)
        f1_c  = f1_score(yc, pred, zero_division=0)
        auroc_c = roc_auc_score(yc, pc) if yc.nunique() > 1 else float("nan")
        ini_str = "-"
        if inic is not None:
            det = int(pred[inic == 1].sum())
            tot = int(inic.sum())
            ini_str = f"{det}/{tot}"
        print(f"{mname:<15} {rec:.3f}   {prec:.3f}  {f1_c:.3f}  {auroc_c:.3f}  {ini_str}")


=== Bucaramanga (68001) | n=23 | objetivo=74% ===
Modelo          Recall  Prec   F1     AUROC  Inicios
XGBoost         0.941   0.941  0.941  0.990  0/1
LightGBM        0.941   0.941  0.941  0.990  0/1
RF              1.000   0.944  0.971  0.990  1/1
Logistica       0.941   0.889  0.914  0.912  0/1
GAM             0.941   0.762  0.842  0.931  0/1

=== Cali (76001) | n=23 | objetivo=48% ===
Modelo          Recall  Prec   F1     AUROC  Inicios
XGBoost         1.000   0.917  0.957  1.000  0/0
LightGBM        1.000   0.917  0.957  1.000  0/0
RF              1.000   0.917  0.957  1.000  0/0
Logistica       1.000   1.000  1.000  1.000  0/0
GAM             0.909   0.833  0.870  0.909  0/0


In [ ]:
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import pickle, os

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
MODEL_DIR = os.path.join("..", "model")

for ax, (name, raw_pkl, cal_pkl) in zip(axes, [
    ("XGBoost",  "xgb_clasico.pkl",  "xgb_clasico_calibrated.pkl"),
    ("LightGBM", "lgbm_clasico.pkl", "lgbm_clasico_calibrated.pkl"),
]):
    with open(os.path.join(MODEL_DIR, raw_pkl), "rb") as fh:
        raw_m = pickle.load(fh)
    with open(os.path.join(MODEL_DIR, cal_pkl), "rb") as fh:
        cal_d = pickle.load(fh)

    feats_raw = list(raw_m.feature_names_in_)
    prob_raw  = raw_m.predict_proba(X_test[feats_raw].fillna(0))[:, 1]
    prob_cal  = cal_d["calibrator"].transform(prob_raw)

    frac_raw, mean_raw = calibration_curve(y_test, prob_raw, n_bins=10)
    frac_cal, mean_cal = calibration_curve(y_test, prob_cal, n_bins=10)

    ax.plot([0, 1], [0, 1], "k--", label="Perfecta")
    ax.plot(mean_raw, frac_raw, "o-", label="Sin calibrar")
    ax.plot(mean_cal, frac_cal, "s-", label="Calibrado (isotonic)")
    ax.set_xlabel("Probabilidad predicha")
    ax.set_ylabel("Fraccion positivos")
    ax.set_title(f"Calibracion — {name}")
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle("Curvas de calibracion T+1 (test 2024-2025)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import os

REPORT_DIR = os.path.join("..", "reports")
CITIES = {"68001": "Bucaramanga", "76001": "Cali"}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (div, city) in zip(axes, CITIES.items()):
    png = os.path.join(REPORT_DIR, f"shap_local_{div}_T1.png")
    if os.path.exists(png):
        img = mpimg.imread(png)
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(f"SHAP local — {city} (T+1)", fontsize=11)
    else:
        ax.text(0.5, 0.5, f"PNG no encontrado:\n{png}", ha="center", va="center")
        ax.axis("off")

plt.suptitle("SHAP local por ciudad (TreeExplainer XGBoost T+1)", y=1.01)
plt.tight_layout()
plt.show()

# Top 5 features per city from parquet
shap_df = pd.read_parquet(os.path.join("..", "model", "shap_local_T1.parquet"))
for div, city in CITIES.items():
    mask = shap_df["divipola"].astype(str) == div
    sv   = shap_df[mask][[c for c in shap_df.columns if c.startswith("shap_")]].values
    feat_names = [c.replace("shap_", "") for c in shap_df.columns if c.startswith("shap_")]
    top5 = sorted(zip(feat_names, np.abs(sv).mean(axis=0)), key=lambda x: -x[1])[:5]
    print(f"\n{city} ({div}) — top 5 SHAP local (mean |SHAP|):")
    for feat, val in top5:
        print(f"  {feat:<35} {val:.4f}")